# Trajectory Timing Analysis

Set `TRAJECTORY_DIR` to any run folder under `trajectories/`, then run the notebook.

Outputs:
- `timings[method]` — dict of timing arrays keyed by dotted path (e.g. `"confidence_time.indirect_time"`)
- `by_method`, `by_problem`, `by_key` — the usual structural lookups
- Quick summary table printed per method

In [10]:
from collections import defaultdict
from pathlib import Path
from pprint import pprint
import json
import re

try:
    import pandas as pd
except ImportError:
    pd = None


# ============ CONFIGURE THESE ============

TRAJECTORY_DIR_CACHE = Path("trajectories/test-llama-bm-622-batch-cache_llama_bigbench_movie_s2")
TRAJECTORY_DIR_NOCACHE = Path("trajectories/test-llama-bm-622-batch-nocache_llama_bigbench_movie_s2")

TRAJECTORY_DIR_CACHE = Path("trajectories/test-llama-bm-622-serial-cache_llama_bigbench_movie_s2")
TRAJECTORY_DIR_NOCACHE = Path("trajectories/test-llama-bm-622-serial-nocache_llama_bigbench_movie_s2")
# =========================================

METHODS = ['vanilla', 'rejection', 'lawyer', 'stepbootstrap']

TRAJ_FILE_RE = re.compile(
    r"^traj_(?P<index>\d+)(?:_sample_(?P<sample>\d+))?(?P<error>_error)?\.json$"
)


def freeze_defaultdict(value):
    if isinstance(value, defaultdict):
        value = dict(value)
    if isinstance(value, dict):
        return {key: freeze_defaultdict(item) for key, item in value.items()}
    if isinstance(value, list):
        return [freeze_defaultdict(item) for item in value]
    return value


def parse_trajectory_filename(path):
    match = TRAJ_FILE_RE.match(path.name)
    if match is None:
        return {"index": None, "sample": None, "is_error": False}
    sample = match.group("sample")
    return {
        "index": int(match.group("index")),
        "sample": int(sample) if sample is not None else None,
        "is_error": bool(match.group("error")),
    }


def method_for(path, root):
    relative = path.relative_to(root)
    return relative.parts[0] if len(relative.parts) > 1 else "root"


def load_trajectory_folder(folder):
    root = Path(folder)
    if not root.exists():
        raise FileNotFoundError(f"Trajectory folder does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"Expected a directory: {root}")

    records = []
    errors = []

    for path in sorted(root.rglob("*.json")):
        with path.open("r", encoding="utf-8") as handle:
            data = json.load(handle)

        meta = parse_trajectory_filename(path)
        entry = {
            **meta,
            "method": method_for(path, root),
            "path": path,
            "relative_path": path.relative_to(root),
            "data": data,
        }
        records.append(entry)
        if meta["is_error"] or "error_type" in data:
            errors.append(entry)

    by_method = defaultdict(list)
    raw_data = defaultdict(lambda: defaultdict(dict))
    by_problem = defaultdict(lambda: defaultdict(dict))
    by_key = {}

    for entry in records:
        m = entry["method"]
        index = entry["index"]
        sample = entry["sample"]
        data = entry["data"]
        by_method[m].append(entry)
        raw_data[m][index][sample] = data
        by_problem[index][m][sample] = data
        by_key[(m, index, sample)] = data

    return {
        "root": root,
        "records": records,
        "errors": errors,
        "by_method": freeze_defaultdict(by_method),
        "raw_data": freeze_defaultdict(raw_data),
        "by_problem": freeze_defaultdict(by_problem),
        "by_key": by_key,
    }


def collect_timings(by_method):
    """Walk every trajectory's 'timings' dict and flatten into arrays by dotted key path."""
    timings = {}
    for method, entries in by_method.items():
        arrays = defaultdict(list)
        for entry in entries:
            t = entry["data"].get("timings")
            if t is None:
                continue
            _flatten_into(t, "", arrays)
        timings[method] = dict(arrays)
    return timings


def _flatten_into(obj, prefix, arrays):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == "debug":
                _flatten_into(v, prefix + "debug.", arrays)
            elif isinstance(v, (int, float)):
                arrays[prefix + k].append(v)
            elif isinstance(v, dict):
                _flatten_into(v, prefix + k + ".", arrays)
    elif isinstance(obj, (int, float)):
        arrays[prefix.rstrip(".")].append(obj)

In [11]:
# Load cache
loaded_cache = load_trajectory_folder(TRAJECTORY_DIR_CACHE)
by_method_cache = loaded_cache["by_method"]
by_problem_cache = loaded_cache["by_problem"]
by_key_cache = loaded_cache["by_key"]

print(f"[CACHE] Loaded {len(loaded_cache['records'])} files from {loaded_cache['root']}")
print(f"  Errors: {len(loaded_cache['errors'])}")
for method, entries in sorted(by_method_cache.items()):
    print(f"    {method}: {len(entries)}")

# Load nocache
loaded_nocache = load_trajectory_folder(TRAJECTORY_DIR_NOCACHE)
by_method_nocache = loaded_nocache["by_method"]
by_problem_nocache = loaded_nocache["by_problem"]
by_key_nocache = loaded_nocache["by_key"]

print(f"\n[NOCACHE] Loaded {len(loaded_nocache['records'])} files from {loaded_nocache['root']}")
print(f"  Errors: {len(loaded_nocache['errors'])}")
for method, entries in sorted(by_method_nocache.items()):
    print(f"    {method}: {len(entries)}")

timings_cache = collect_timings(by_method_cache)
timings_nocache = collect_timings(by_method_nocache)

[CACHE] Loaded 16 files from trajectories/test-llama-bm-622-serial-cache_llama_bigbench_movie_s2
  Errors: 0
    lawyer: 4
    rejection: 4
    stepbootstrap: 6
    vanilla: 2

[NOCACHE] Loaded 16 files from trajectories/test-llama-bm-622-serial-nocache_llama_bigbench_movie_s2
  Errors: 0
    lawyer: 4
    rejection: 4
    stepbootstrap: 6
    vanilla: 2


In [12]:
# Side-by-side summary: cache vs nocache
import statistics

all_methods = sorted(set(list(timings_cache.keys()) + list(timings_nocache.keys())))
all_keys = sorted(set(
    k for t in (timings_cache, timings_nocache) for arrays in t.values() for k in arrays
))

for method in all_methods:
    cache_arrays = timings_cache.get(method, {})
    nocache_arrays = timings_nocache.get(method, {})
    n_cache = len(next(iter(cache_arrays.values()), []))
    n_nocache = len(next(iter(nocache_arrays.values()), []))
    print(f"\n{'='*80}")
    print(f"  {method.upper()}  (cache: {n_cache}, nocache: {n_nocache} trajectories)")
    print(f"{'='*80}")
    print(f"  {'key':40s}  {'cache mean':>12s}  {'nocache mean':>12s}  {'speedup':>8s}")
    print(f"  {'-'*40}  {'-'*12}  {'-'*12}  {'-'*8}")
    for key in all_keys:
        c_vals = cache_arrays.get(key, [])
        nc_vals = nocache_arrays.get(key, [])
        if not c_vals and not nc_vals:
            continue
        c_mean = statistics.mean(c_vals) if c_vals else float('nan')
        nc_mean = statistics.mean(nc_vals) if nc_vals else float('nan')
        if nc_mean and c_mean and nc_mean != 0:
            speedup = f"{nc_mean / c_mean:.2f}x"
        else:
            speedup = "—"
        c_str = f"{c_mean:.6f}s" if c_vals else "—"
        nc_str = f"{nc_mean:.6f}s" if nc_vals else "—"
        print(f"  {key:40s}  {c_str:>12s}  {nc_str:>12s}  {speedup:>8s}")


  LAWYER  (cache: 4, nocache: 4 trajectories)
  key                                         cache mean  nocache mean   speedup
  ----------------------------------------  ------------  ------------  --------
  confidence_time.answer_ent_time              0.000298s     0.000186s     0.62x
  confidence_time.answer_prob_time             0.000015s     0.000012s     0.78x
  confidence_time.answer_score_ent_time        0.000651s     0.000564s     0.87x
  confidence_time.answer_score_prob_time       0.000031s     0.000023s     0.75x
  confidence_time.debug.indirect_time_debug.align_cache_time     0.004926s     0.000011s     0.00x
  confidence_time.debug.indirect_time_debug.forward_time     0.023978s     0.029386s     1.23x
  confidence_time.debug.indirect_time_debug.reused_num_tokens   519.500000s     0.000000s         —
  confidence_time.debug.indirect_time_debug.tokenizer_time     0.000180s     0.000201s     1.12x
  confidence_time.debug.verbconf_time_debug.align_cache_time     0.003693s  

In [5]:
# Access individual timing arrays like:
#   timings_cache['stepbootstrap']['confidence_time.indirect_time']
#   timings_nocache['vanilla']['generation_time']
#
# Available keys per method:
for method in sorted(set(list(timings_cache.keys()) + list(timings_nocache.keys()))):
    keys_c = set(timings_cache.get(method, {}).keys())
    keys_nc = set(timings_nocache.get(method, {}).keys())
    print(f"\n{method}: {sorted(keys_c | keys_nc)}")


lawyer: ['confidence_time.answer_ent_time', 'confidence_time.answer_prob_time', 'confidence_time.answer_score_ent_time', 'confidence_time.answer_score_prob_time', 'confidence_time.indirect_time', 'confidence_time.verbconf_time', 'generation_time', 'total_confidence_time']

rejection: ['confidence_time.answer_ent_time', 'confidence_time.answer_prob_time', 'confidence_time.answer_score_ent_time', 'confidence_time.answer_score_prob_time', 'confidence_time.indirect_time', 'confidence_time.verbconf_time', 'generation_time', 'total_confidence_time']

stepbootstrap: ['confidence_time.answer_ent_time', 'confidence_time.answer_prob_time', 'confidence_time.answer_score_ent_time', 'confidence_time.answer_score_prob_time', 'confidence_time.indirect_time', 'confidence_time.verbconf_time', 'generation_time', 'total_confidence_time']

vanilla: ['confidence_time.answer_ent_time', 'confidence_time.answer_prob_time', 'confidence_time.answer_score_ent_time', 'confidence_time.answer_score_prob_time', 'co